[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcosta/generative-networks/blob/main/pretraining/mae.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Masked Autoencoders on CIFAR-10

We implement the asymmetric MAE recipe: remove most image patches before the encoder, reconstruct normalized pixels with a lightweight decoder, discard that decoder, and test the frozen encoder with a linear classifier.

The model in this notebook is deliberately small. It demonstrates a pretraining objective and representation-transfer workflow; it is **not** a foundation model. Foundation models additionally require broad data, much larger capacity, and extensive compute.


We will:

- split images into non-overlapping patches and mask 75% of them;
- send only visible patches through a tiny Vision Transformer;
- restore the original order with mask tokens in a smaller decoder;
- apply mean-squared error only to masked patches;
- visualize reconstruction and compare random and pretrained frozen features.


## 0. Dependencies

PyTorch is not pinned so Colab keeps its CUDA-compatible build.


In [ ]:
from importlib import util
import subprocess
import sys

REQUIRED = {"torch": "torch", "torchvision": "torchvision", "matplotlib": "matplotlib", "numpy": "numpy"}
missing = [package for module, package in REQUIRED.items() if util.find_spec(module) is None]
if missing:
    print("Installing missing course dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Course dependencies are already installed.")


## 1. Imports and Reproducibility


In [ ]:
import copy
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"
print(f"Using device: {device}; FAST_DEV_RUN={fast_dev_run}")


## 2. Configuration

The normal run uses 20 epochs over 20,000 unlabeled images. Smoke mode preserves the architecture but trains two mini-batches.


In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "pretraining" else Path.cwd()
output_dir = project_root / "pretraining" / "outputs" / "mae"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "tiny_mae.pt"

image_size, patch_size = 32, 4
number_of_patches = (image_size // patch_size) ** 2
mask_ratio = 0.75
batch_size = 64 if fast_dev_run else 256
epochs = 1 if fast_dev_run else 20
max_steps = 2 if fast_dev_run else None
learning_rate = 1.5e-3
print("Artifacts:", output_dir)


## 3. CIFAR-10 and Deterministic Evaluation Splits


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
data_root = project_root / "data"

evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
raw_train = datasets.CIFAR10(data_root, train=True, download=True)
probe_source = datasets.CIFAR10(data_root, train=True, transform=evaluation_transform)
test_source = datasets.CIFAR10(data_root, train=False, download=True, transform=evaluation_transform)

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(raw_train), generator=generator).tolist()
pretrain_count = 256 if fast_dev_run else 20_000
probe_count = 256 if fast_dev_run else 10_000
test_count = 256 if fast_dev_run else len(test_source)
pretrain_indices = permutation[:pretrain_count]
probe_indices = permutation[:probe_count]

probe_train = Subset(probe_source, probe_indices)
probe_test = Subset(test_source, range(test_count))
loader_kwargs = {"num_workers": 0 if fast_dev_run else 2, "pin_memory": device.type == "cuda"}
probe_train_loader = DataLoader(probe_train, batch_size=256, shuffle=False, **loader_kwargs)
probe_test_loader = DataLoader(probe_test, batch_size=256, shuffle=False, **loader_kwargs)

print(f"Unlabeled pretraining images: {pretrain_count:,}")
print(f"Labeled probe images: {probe_count:,}; test images: {test_count:,}")


In [ ]:
pretrain_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
pretrain_source = datasets.CIFAR10(data_root, train=True, transform=pretrain_transform)
pretrain_loader = DataLoader(
    Subset(pretrain_source, pretrain_indices), batch_size=batch_size, shuffle=True,
    drop_last=True, generator=torch.Generator().manual_seed(SEED), **loader_kwargs,
)


## 4. Patchification and Random Masking

MAE differs from replacing patches with visible mask tokens before the encoder. The encoder receives only the 25% retained tokens. Mask tokens are introduced after encoding, when the decoder restores the complete sequence.


In [ ]:
def patchify(images):
    batch, channels, height, width = images.shape
    patches = images.reshape(batch, channels, height // patch_size, patch_size, width // patch_size, patch_size)
    return patches.permute(0, 2, 4, 1, 3, 5).reshape(batch, number_of_patches, -1)

def unpatchify(patches):
    batch = patches.size(0)
    grid = image_size // patch_size
    images = patches.reshape(batch, grid, grid, 3, patch_size, patch_size)
    return images.permute(0, 3, 1, 4, 2, 5).reshape(batch, 3, image_size, image_size)

def random_mask(batch, tokens, ratio, current_device):
    keep = int(tokens * (1 - ratio))
    noise = torch.rand(batch, tokens, device=current_device)
    ids_shuffle = noise.argsort(dim=1)
    ids_restore = ids_shuffle.argsort(dim=1)
    ids_keep = ids_shuffle[:, :keep]
    mask = torch.ones(batch, tokens, device=current_device)
    mask[:, :keep] = 0
    mask = mask.gather(1, ids_restore)
    ordered = torch.arange(tokens, device=current_device).expand(batch, -1)
    assert torch.equal(ordered.gather(1, ids_shuffle).gather(1, ids_restore), ordered)
    assert torch.all(mask.sum(1) == tokens - keep)
    return ids_keep, ids_restore, mask

sample_images, _ = next(iter(pretrain_loader))
assert torch.allclose(unpatchify(patchify(sample_images)), sample_images)
print("Patch tensor:", patchify(sample_images).shape)


## 5. Tiny Asymmetric MAE


In [ ]:
class TinyMAE(nn.Module):
    def __init__(self, encoder_dim=128, decoder_dim=64):
        super().__init__()
        patch_dim = 3 * patch_size * patch_size
        self.patch_embedding = nn.Linear(patch_dim, encoder_dim)
        self.encoder_position = nn.Parameter(torch.randn(1, number_of_patches, encoder_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(encoder_dim, 4, encoder_dim * 4, dropout=0.0, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, 4)
        self.encoder_norm = nn.LayerNorm(encoder_dim)
        self.to_decoder = nn.Linear(encoder_dim, decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))
        self.decoder_position = nn.Parameter(torch.randn(1, number_of_patches, decoder_dim) * 0.02)
        decoder_layer = nn.TransformerEncoderLayer(decoder_dim, 4, decoder_dim * 4, dropout=0.0, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerEncoder(decoder_layer, 2)
        self.decoder_norm = nn.LayerNorm(decoder_dim)
        self.pixel_prediction = nn.Linear(decoder_dim, patch_dim)

    def encode(self, images):
        tokens = self.patch_embedding(patchify(images)) + self.encoder_position
        return self.encoder_norm(self.encoder(tokens)).mean(1)

    def forward(self, images):
        patches = patchify(images)
        ids_keep, ids_restore, mask = random_mask(images.size(0), number_of_patches, mask_ratio, images.device)
        tokens = self.patch_embedding(patches)
        visible = tokens.gather(1, ids_keep.unsqueeze(-1).expand(-1, -1, tokens.size(-1)))
        positions = self.encoder_position.expand(images.size(0), -1, -1)
        visible = visible + positions.gather(1, ids_keep.unsqueeze(-1).expand(-1, -1, positions.size(-1)))
        encoded = self.encoder_norm(self.encoder(visible))
        decoded_visible = self.to_decoder(encoded)
        missing = self.mask_token.expand(images.size(0), number_of_patches - decoded_visible.size(1), -1)
        shuffled = torch.cat([decoded_visible, missing], dim=1)
        restored = shuffled.gather(1, ids_restore.unsqueeze(-1).expand(-1, -1, shuffled.size(-1)))
        decoded = self.decoder_norm(self.decoder(restored + self.decoder_position))
        prediction = self.pixel_prediction(decoded)
        mean = patches.mean(-1, keepdim=True)
        variance = patches.var(-1, keepdim=True, unbiased=False)
        targets = (patches - mean) / torch.sqrt(variance + 1e-6)
        per_patch = (prediction - targets).square().mean(-1)
        loss = (per_patch * mask).sum() / mask.sum()
        assert torch.allclose(loss, per_patch[mask.bool()].mean())
        return loss, prediction, mask, mean, variance

    def forward_features(self, images):
        return self.encode(images)

model = TinyMAE().to(device)
images = sample_images[:4].to(device)
loss, prediction, mask, _, _ = model(images)
loss.backward()
assert math.isfinite(loss.item()) and model.patch_embedding.weight.grad is not None
model.zero_grad(set_to_none=True)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M; initial loss={loss.item():.3f}")


## 6. Pretrain the MAE


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)
history, step = [], 0
for epoch in range(epochs):
    model.train()
    running = []
    for images, _ in pretrain_loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            loss, *_ = model(images)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running.append(loss.item())
        step += 1
        if max_steps is not None and step >= max_steps:
            break
    history.append(float(np.mean(running)))
    print(f"Epoch {epoch + 1:02d}/{epochs:02d} | masked MSE={history[-1]:.4f}")
    if max_steps is not None and step >= max_steps:
        break

torch.save({"model": model.state_dict(), "history": history, "config": {"patch_size": patch_size, "mask_ratio": mask_ratio}}, checkpoint_path)
assert np.isfinite(history).all()
plt.plot(range(1, len(history) + 1), history, marker="o")
plt.xlabel("Epoch"); plt.ylabel("Masked-patch MSE"); plt.title("MAE pretraining"); plt.grid(alpha=0.3)


## 7. Visualize Masking and Reconstruction


In [ ]:
model.eval()
with torch.no_grad():
    shown = sample_images[:8].to(device)
    _, prediction, mask, mean, variance = model(shown)
    raw_prediction = prediction * torch.sqrt(variance + 1e-6) + mean
    original_patches = patchify(shown)
    visible = original_patches * (1 - mask.unsqueeze(-1))
    reconstruction = original_patches * (1 - mask.unsqueeze(-1)) + raw_prediction * mask.unsqueeze(-1)
    display_images = torch.cat([shown, unpatchify(visible), unpatchify(reconstruction)]).cpu()
    display_images = display_images * torch.tensor(CIFAR_STD).view(1, 3, 1, 1) + torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
plt.figure(figsize=(12, 5)); plt.imshow(make_grid(display_images.clamp(0, 1), nrow=8).permute(1, 2, 0)); plt.axis("off")
plt.title("Originals (top), visible 25% (middle), MAE reconstructions (bottom)")


## 8. Frozen Linear Probing

The decoder is no longer used. A linear probe measures information accessible from the frozen encoder without adapting its parameters.


In [ ]:
@torch.no_grad()
def extract_features(encoder, loader):
    encoder.eval()
    features, labels = [], []
    for images, targets in loader:
        features.append(encoder(images.to(device)).float().cpu())
        labels.append(targets)
    return torch.cat(features), torch.cat(labels)

def fit_linear_probe(encoder, name):
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    train_x, train_y = extract_features(encoder, probe_train_loader)
    test_x, test_y = extract_features(encoder, probe_test_loader)
    head = nn.Linear(train_x.size(1), 10).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
    train_loader = DataLoader(
        TensorDataset(train_x, train_y), batch_size=256, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    epochs = 1 if fast_dev_run else 15
    history = []
    for epoch in range(epochs):
        head.train()
        for features, targets in train_loader:
            logits = head(features.to(device))
            loss = F.cross_entropy(logits, targets.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.no_grad():
            predictions = head(test_x.to(device)).argmax(1).cpu()
        accuracy = (predictions == test_y).float().mean().item()
        history.append(accuracy)
    print(f"{name}: final frozen-probe accuracy={accuracy:.3f}")
    return {"name": name, "accuracy": accuracy, "history": history}


In [ ]:
class MAEFeatureEncoder(nn.Module):
    def __init__(self, mae):
        super().__init__(); self.mae = mae
    def forward(self, images):
        return self.mae.encode(images)

random_encoder = MAEFeatureEncoder(TinyMAE())
pretrained_encoder = MAEFeatureEncoder(model)


In [ ]:
probe_results = [
    fit_linear_probe(random_encoder.to(device), "Random frozen encoder"),
    fit_linear_probe(pretrained_encoder.to(device), "Pretrained frozen encoder"),
]
plt.figure(figsize=(6, 4))
plt.bar([row["name"] for row in probe_results], [row["accuracy"] for row in probe_results])
plt.ylabel("CIFAR-10 test accuracy")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1)
plt.title("Representation quality through a linear probe")
plt.tight_layout()


## 9. Reload the Checkpoint


In [ ]:
reloaded = TinyMAE().to(device)
reloaded.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True)["model"])
reloaded.eval(); model.eval()
with torch.no_grad():
    assert torch.equal(model.encode(images), reloaded.encode(images))
print("Reloaded encoder features match exactly.")


## 10. Things to Try

- Sweep mask ratios from 50% to 90%.
- Compare normalized-pixel targets with raw-pixel targets.
- Increase decoder depth without changing the encoder.
- Fine-tune the complete encoder after linear probing.

## Troubleshooting

- Use a GPU for the normal run; `FAST_DEV_RUN=1` is intended for pipeline checks.
- If memory is exhausted, lower `batch_size`; do not reduce the mask ratio as a memory workaround.
- Blurry reconstructions are expected from a compact model and short pretraining run.

**Reference:** [Masked Autoencoders Are Scalable Vision Learners](https://arxiv.org/abs/2111.06377).
